# GARTOK style LoRA (SD 1.5)
Trains a style LoRA on the game's own portraits (230 images, 200x200, trigger word `gartokstyle`).
Use the **T4 GPU** runtime (Runtime > Change runtime type). Expect ~1 h of training.
Run the cells top to bottom. If the install cell asks to restart the session, restart and run it again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install the trainer (kohya sd-scripts)

In [ ]:
%cd /content
!git clone -q https://github.com/kohya-ss/sd-scripts
%cd /content/sd-scripts
!pip install -q -r requirements.txt
!accelerate config default --mixed_precision fp16

## 2. Upload the dataset zip (`gartok_style.zip`, 22 MB)

In [ ]:
from google.colab import files
import os, shutil, zipfile

up = files.upload()
name = next(iter(up))
raw = "/content/raw"
shutil.rmtree(raw, ignore_errors=True)
zipfile.ZipFile(name).extractall(raw)
print(len(os.listdir(raw)) // 2, "images")

## 3. Prepare images
Flattens the transparent corners onto parchment and upscales 200px -> 512px (Lanczos).
The folder name `5_gartokstyle` means: repeat each image 5 times per epoch.

In [ ]:
from PIL import Image

data = "/content/dataset/5_gartokstyle"
shutil.rmtree("/content/dataset", ignore_errors=True)
os.makedirs(data)
for f in sorted(os.listdir(raw)):
    if f.endswith(".png"):
        im = Image.open(f"{raw}/{f}").convert("RGBA")
        bg = Image.new("RGBA", im.size, (236, 224, 196, 255))
        bg.alpha_composite(im)
        bg.convert("RGB").resize((512, 512), Image.LANCZOS).save(f"{data}/{f[:-4]}.png")
    elif f.endswith(".txt"):
        shutil.copy(f"{raw}/{f}", f"{data}/{f}")
print(len(os.listdir(data)) // 2, "images ready")

## 4. Base model

In [ ]:
!wget -q -O /content/v1-5-pruned-emaonly.safetensors https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
!ls -lh /content/v1-5-pruned-emaonly.safetensors

## 5. Train
230 images x 5 repeats x 10 epochs / batch 4 = ~2900 steps. A checkpoint is saved every 2 epochs
(`gartok_style-000002.safetensors`, ... and the final `gartok_style.safetensors`) so you can compare them:
earlier ones are weaker, later ones stronger but may overfit.

In [ ]:
%cd /content/sd-scripts
!accelerate launch --num_cpu_threads_per_process 2 train_network.py   --pretrained_model_name_or_path=/content/v1-5-pruned-emaonly.safetensors   --train_data_dir=/content/dataset   --output_dir=/content/output --output_name=gartok_style --save_model_as=safetensors   --caption_extension=.txt --resolution=512,512 --cache_latents   --network_module=networks.lora --network_dim=32 --network_alpha=16   --train_batch_size=4 --max_train_epochs=10 --save_every_n_epochs=2   --learning_rate=1e-4 --unet_lr=1e-4 --text_encoder_lr=5e-5   --lr_scheduler=cosine_with_restarts --lr_warmup_steps=100   --optimizer_type=AdamW8bit --mixed_precision=fp16 --save_precision=fp16   --sdpa --seed=42

## 6. Download the results
Put the `.safetensors` files in `stable-diffusion-webui/models/Lora/`.

In [ ]:
import glob
for p in sorted(glob.glob("/content/output/*.safetensors")):
    print(p)
    files.download(p)